# LeetCode #173 — Binary Search Tree Iterator: duyệt inorder "theo yêu cầu" bằng ngăn xếp

**Mức độ:** Medium · **Chủ đề:** Stack, Tree, Design, Binary Search Tree, Iterator

Skeleton trên LeetCode:

```python
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
    def next(self) -> int:
    def hasNext(self) -> bool:
```

Mục tiêu của notebook:

1. Hiểu **iterator**: đối tượng trả về từng phần tử một khi được hỏi, thay vì trả cả danh sách.
2. Cách dễ: duyệt inorder trước, lưu vào list — đúng nhưng tốn `O(n)` bộ nhớ.
3. Cách chuẩn: **ngăn xếp chứa "nhánh trái"** — tách bản inorder lặp thành từng bước nhỏ, bộ nhớ `O(h)`.
4. Phân tích **khấu hao** (amortized): mỗi `next()` trung bình `O(1)`.

# 1. Bài toán là gì?

Cài đặt lớp `BSTIterator` duyệt **inorder** (tức là tăng dần) một cây nhị phân tìm kiếm:

- `BSTIterator(root)`: khởi tạo. Con trỏ ban đầu đứng ở một vị trí "trước phần tử nhỏ nhất".
- `next()`: dịch con trỏ sang phải và trả về số tại vị trí mới (lần gọi đầu trả số **nhỏ nhất**).
- `hasNext()`: còn số bên phải con trỏ không.

Đề đảm bảo `next()` luôn hợp lệ khi được gọi.

**Constraints (theo đề):**

- Số nút trong khoảng `[1, 10^5]`.
- `0 <= Node.val <= 10^6`
- Tối đa `10^5` lời gọi `hasNext` và `next`.

**Follow up (theo đề):** `next()` và `hasNext()` chạy trung bình `O(1)` và dùng bộ nhớ `O(h)` (h là chiều cao cây)?

---

## Ví dụ

```text
Cây:        7
           / \
          3   15
             /  \
            9    20

["BSTIterator","next","next","hasNext","next","hasNext","next","hasNext","next","hasNext"]
-> [null, 3, 7, true, 9, true, 15, true, 20, false]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân tìm kiếm (BST)
2. Ngăn xếp (stack)
3. Iterator và generator trong Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân tìm kiếm (BST)

### BST là gì?

Cây nhị phân tìm kiếm (Binary Search Tree) là cây nhị phân thỏa, **với mọi nút**:

- mọi giá trị trong cây con **trái** `<` giá trị nút,
- mọi giá trị trong cây con **phải** `>` giá trị nút.

```text
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13
```

Chú ý: điều kiện áp dụng cho **toàn bộ** cây con, không chỉ con trực tiếp (ví dụ `7` nằm bên trái `8` nên phải `< 8`).

### Ba tính chất quan trọng

1. **Tìm kiếm** giống tìm nhị phân: so với nút, nhỏ hơn thì sang trái, lớn hơn thì sang phải → `O(h)` (h = chiều cao).
2. **Duyệt inorder cho dãy tăng dần**: `1 3 4 6 7 8 10 13 14`.
3. Cây **cân bằng** có `h ≈ log₂ n`; cây lệch có `h = n` (tệ như danh sách).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def bst_chen(root, x):
    if root is None:
        return TreeNode(x)
    if x < root.val:
        root.left = bst_chen(root.left, x)
    else:
        root.right = bst_chen(root.right, x)
    return root

def bst_tim(root, x):
    buoc = 0
    while root:
        buoc += 1
        if x == root.val:
            return True, buoc
        root = root.left if x < root.val else root.right
    return False, buoc

def bst_inorder(node, out):
    if node:
        bst_inorder(node.left, out); out.append(node.val); bst_inorder(node.right, out)

r = None
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    r = bst_chen(r, x)
out = []; bst_inorder(r, out)
print("inorder:", out)
print("tìm 7:", bst_tim(r, 7), "| tìm 5:", bst_tim(r, 5), "(đã tìm, số bước)")

inorder: [1, 3, 4, 6, 7, 8, 10, 13, 14]
tìm 7: (True, 4) | tìm 5: (False, 4) (đã tìm, số bước)


### Dựng BST cân bằng từ dãy đã sắp xếp

Lấy phần tử **giữa** làm gốc, nửa trái dựng cây con trái, nửa phải dựng cây con phải (chia để trị). Chiều cao chỉ `O(log n)`.

In [2]:
def bst_tu_mang(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(a[mid], bst_tu_mang(a, lo, mid - 1), bst_tu_mang(a, mid + 1, hi))

def cao(n):
    return 0 if n is None else 1 + max(cao(n.left), cao(n.right))

lech = None
for x in range(1, 16):
    lech = bst_chen(lech, x)                  # chèn tăng dần → cây lệch
print("15 số chèn tăng dần: cao", cao(lech), "| dựng từ giữa: cao", cao(bst_tu_mang(list(range(1, 16)))))

15 số chèn tăng dần: cao 15 | dựng từ giữa: cao 4


### Kiểm tra một cây có phải BST

Sai lầm phổ biến: chỉ so mỗi nút với **con trực tiếp**. Đúng: truyền xuống **khoảng cho phép** `(lo, hi)`; sang trái thì cận trên thành giá trị nút, sang phải thì cận dưới thành giá trị nút. (Hoặc: inorder phải tăng ngặt.)

In [3]:
def la_bst(node, lo=float("-inf"), hi=float("inf")):
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return la_bst(node.left, lo, node.val) and la_bst(node.right, node.val, hi)

def chi_so_con(node):                          # cách SAI
    if node is None:
        return True
    if node.left and node.left.val >= node.val: return False
    if node.right and node.right.val <= node.val: return False
    return chi_so_con(node.left) and chi_so_con(node.right)

bay = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))   # 4 nằm bên phải 5 → không phải BST
print("đúng:", la_bst(bay), "| cách chỉ so con trực tiếp (sai):", chi_so_con(bay))

đúng: False | cách chỉ so con trực tiếp (sai): True


## 2.2 Ngăn xếp (stack)

### Ngăn xếp là gì?

Ngăn xếp là cấu trúc **vào sau, ra trước** (LIFO — Last In, First Out), giống chồng đĩa: chỉ đặt thêm hoặc lấy ra ở **đỉnh**.

```text
push 1   push 2   push 3   pop → 3   top = 2
                   │ 3 │
          │ 2 │    │ 2 │    │ 2 │
 │ 1 │    │ 1 │    │ 1 │    │ 1 │
 └───┘    └───┘    └───┘    └───┘
```

| Thao tác | Ý nghĩa | Python (`list`) | Độ phức tạp |
|---|---|---|---|
| `push(x)` | đặt `x` lên đỉnh | `st.append(x)` | `O(1)` |
| `pop()` | lấy và bỏ đỉnh | `st.pop()` | `O(1)` |
| `top()` | xem đỉnh | `st[-1]` | `O(1)` |
| rỗng? | | `not st` | `O(1)` |

In [4]:
st = []
for x in [1, 2, 3]:
    st.append(x)
    print("push", x, "->", st)
print("top:", st[-1])
while st:
    print("pop", st.pop(), "->", st)
try:
    st.pop()
except IndexError as e:
    print("pop khi rỗng:", e)

push 1 -> [1]
push 2 -> [1, 2]
push 3 -> [1, 2, 3]
top: 3
pop 3 -> [1, 2]
pop 2 -> [1]
pop 1 -> []
pop khi rỗng: pop from empty list


### Khi nào nghĩ tới ngăn xếp?

Khi thứ được mở **sau cùng** phải được đóng/xử lý **trước tiên**:

- Dấu ngoặc lồng nhau: `( [ { } ] )`.
- Đường dẫn thư mục: `..` hủy thư mục vừa vào.
- Tính biểu thức hậu tố, gọi hàm đệ quy (máy tính dùng *call stack*).
- "Hoàn tác" (undo).

Ví dụ kinh điển — kiểm tra ngoặc hợp lệ: gặp ngoặc mở thì push; gặp ngoặc đóng thì đỉnh phải là ngoặc mở tương ứng.

In [5]:
def ngoac_hop_le(s):
    cap = {")": "(", "]": "[", "}": "{"}
    st = []
    for ch in s:
        if ch in "([{":
            st.append(ch)
        else:
            if not st or st[-1] != cap[ch]:
                return False
            st.pop()
    return not st                        # còn ngoặc mở chưa đóng → sai

for s in ["([]{})", "([)]", "((", "}"]:
    print(f"{s!r:>9}: {ngoac_hop_le(s)}")

 '([]{})': True
   '([)]': False
     '((': False
      '}': False


### Lỗi hay gặp

- `pop()` / `st[-1]` khi ngăn xếp rỗng → `IndexError`. Luôn kiểm tra `if st`.
- Quên kiểm tra ngăn xếp **còn sót** phần tử ở cuối.
- Dùng `list.pop(0)` / `insert(0, x)` (đầu danh sách) — đó là hàng đợi và tốn `O(n)`; ngăn xếp luôn thao tác ở **cuối** list.

## 2.3 Iterator và generator trong Python

### Iterator là gì?

Một **iterator** là đối tượng trả về **lần lượt** từng phần tử, theo yêu cầu, qua hai phương thức:

- `__iter__()` trả về chính nó.
- `__next__()` trả về phần tử kế tiếp, hoặc ném `StopIteration` khi hết.

Vòng `for x in obj` thực chất gọi `iter(obj)` rồi `next(...)` liên tục tới khi gặp `StopIteration`.

In [6]:
it = iter([10, 20, 30])
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("hết phần tử → StopIteration")

10 20 30
hết phần tử → StopIteration


### Tự viết một iterator

In [7]:
class CountDown:
    def __init__(self, start):
        self.cur = start
    def __iter__(self):
        return self
    def __next__(self):
        if self.cur <= 0:
            raise StopIteration
        self.cur -= 1
        return self.cur + 1

print(list(CountDown(4)))

[4, 3, 2, 1]


### Generator: iterator viết bằng `yield`

Hàm chứa `yield` trả về một **generator**. Mỗi lần `next`, hàm chạy tiếp tới `yield` kế tiếp rồi **tạm dừng**, giữ nguyên biến cục bộ. Đây là cách ngắn nhất để viết iterator.

In [8]:
def count_down_gen(start):
    while start > 0:
        yield start
        start -= 1

g_demo = count_down_gen(3)
print(next(g_demo), list(g_demo))

3 [2, 1]


### Lười (lazy): chỉ tính khi được hỏi

Iterator không tạo sẵn toàn bộ danh sách → bộ nhớ chỉ tỉ lệ với **trạng thái** cần giữ, không phải tổng số phần tử. Có thể dừng giữa chừng mà không tốn công cho phần còn lại.

In [9]:
import itertools, sys
squares = (i * i for i in itertools.count(1))         # vô hạn, nhưng không tốn bộ nhớ
print("5 bình phương đầu:", list(itertools.islice(squares, 5)))
print("kích thước list 10^6 phần tử:", sys.getsizeof(list(range(10**6))), "byte | generator:", sys.getsizeof(x for x in range(10**6)), "byte")

5 bình phương đầu: [1, 4, 9, 16, 25]
kích thước list 10^6 phần tử: 8000056 byte | generator: 192 byte


### Thiết kế kiểu `hasNext()` / `next()`

LeetCode thường yêu cầu iterator kiểu Java: `hasNext()` trả lời "còn phần tử không" **mà không lấy ra**, `next()` lấy ra. Cách làm phổ biến: giữ **trạng thái** đủ để biết phần tử kế tiếp (con trỏ, ngăn xếp…), hoặc "nhìn trước" một phần tử.

In [10]:
class PeekIterDemo:
    def __init__(self, iterable):
        self._it = iter(iterable)
        self._buf = []                          # tối đa 1 phần tử đã nhìn trước
    def hasNext(self):
        if not self._buf:
            try:
                self._buf.append(next(self._it))
            except StopIteration:
                return False
        return True
    def next(self):
        self.hasNext()
        return self._buf.pop()

p_demo = PeekIterDemo("ab")
print(p_demo.hasNext(), p_demo.hasNext(), p_demo.next(), p_demo.next(), p_demo.hasNext())

True

 True a b False


### Lỗi hay gặp

- `hasNext()` làm **mất** phần tử (gọi `next` mà không lưu lại).
- Dùng lại một iterator đã cạn → không còn phần tử (`list(g)` lần hai trả `[]`).
- Tạo sẵn toàn bộ danh sách trong hàm khởi tạo khi đề yêu cầu bộ nhớ `O(h)` hoặc dữ liệu rất lớn.

## 2.4 Áp dụng vào bài này

Duyệt **inorder** một **BST** cho các giá trị **tăng dần**. Thay vì duyệt sẵn toàn bộ vào một list (`O(n)` bộ nhớ), ta làm một **iterator** "lười": dùng **ngăn xếp** giữ đường đi xuống nhánh trái — chính là duyệt inorder lặp bị "cắt" thành từng bước `next()`. Bộ nhớ chỉ `O(h)`, và trung bình mỗi `next()` là `O(1)` (mỗi nút vào/ra ngăn xếp một lần).

# 3. Nền tảng

## Cây và các hàm tiện ích

In [11]:
from typing import List, Optional
from collections import deque
import random, time, sys


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def inorder_list(node, out=None):
    out = [] if out is None else out
    if node:
        inorder_list(node.left, out); out.append(node.val); inorder_list(node.right, out)
    return out

root = build_tree([7, 3, 15, None, None, 9, 20])
print("inorder:", inorder_list(root))

inorder: [3, 7, 9, 15, 20]


## Iterator trong Python

Python có sẵn khái niệm iterator: `iter(x)` trả về iterator, `next(it)` lấy phần tử kế tiếp, hết thì báo `StopIteration`. **Generator** (hàm có `yield`) là cách viết iterator ngắn nhất: hàm "tạm dừng" ở mỗi `yield` và chạy tiếp khi được hỏi.

In [12]:
it = iter([10, 20, 30])
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("hết phần tử → StopIteration")

def dem_toi(n):
    for i in range(1, n + 1):
        print(f"   (sinh {i})")
        yield i

g = dem_toi(3)
print("lấy:", next(g))
print("lấy:", next(g))

10 20 30
hết phần tử → StopIteration
   (sinh 1)
lấy: 1
   (sinh 2)
lấy: 2


Generator chỉ làm việc **khi được hỏi** — đó chính là tinh thần của bài này: không duyệt cả cây ngay từ đầu.

# 4. Baseline: duyệt hết rồi lưu vào list

In [13]:
class BSTIteratorList:
    def __init__(self, root):
        self.vals = inorder_list(root)    # O(n) bộ nhớ
        self.i = 0

    def next(self) -> int:
        v = self.vals[self.i]
        self.i += 1
        return v

    def hasNext(self) -> bool:
        return self.i < len(self.vals)

it_list = BSTIteratorList(root)
print([it_list.next() for _ in range(5)], it_list.hasNext())

[3, 7, 9, 15, 20] False


`next`/`hasNext` đều `O(1)`, nhưng bộ nhớ `O(n)` — không đạt follow-up `O(h)`. Với cây cân bằng 10^5 nút, `h ≈ 17` trong khi `n = 10^5`.

# 5. Ý tưởng chính: ngăn xếp chứa nhánh trái

Nhắc lại inorder lặp bằng ngăn xếp:

```text
cur = root
while cur or st:
    while cur: st.append(cur); cur = cur.left     # (A) đi trái hết mức
    cur = st.pop(); thăm cur                       # (B) lấy nút nhỏ nhất chưa thăm
    cur = cur.right                                # (C) sang cây con phải
```

Ta **cắt** vòng lặp này thành từng lần `next()`:

- Khởi tạo: làm bước (A) từ `root` → đỉnh ngăn xếp là nút **nhỏ nhất**.
- `next()`: pop nút đỉnh (bước B), rồi làm bước (A) cho cây con **phải** của nó (bước C + A). Trả về giá trị nút vừa pop.
- `hasNext()`: ngăn xếp còn phần tử không.

**Bất biến:** đỉnh ngăn xếp luôn là nút nhỏ nhất chưa được trả về; ngăn xếp chứa đúng các nút "tổ tiên đang chờ" trên đường đi — tối đa `h` nút.

In [14]:
class BSTIteratorStack:
    def __init__(self, root):
        self.st = []
        self._push_left(root)

    def _push_left(self, node):
        while node:
            self.st.append(node)
            node = node.left

    def next(self) -> int:
        node = self.st.pop()
        self._push_left(node.right)
        return node.val

    def hasNext(self) -> bool:
        return bool(self.st)

it_st = BSTIteratorStack(root)
out = []
while it_st.hasNext():
    out.append(it_st.next())
print(out)

[3, 7, 9, 15, 20]


# 6. Mô phỏng

In [15]:
def trace_iterator(values):
    r = build_tree(values)
    it = BSTIteratorStack(r)
    print(f"khởi tạo: ngăn xếp = {[n.val for n in it.st]}")
    while it.hasNext():
        top = it.st[-1].val
        v = it.next()
        print(f"next() → {v:>2} | pop {top}, đẩy nhánh trái của cây con phải → ngăn xếp = {[n.val for n in it.st]}")
    print("hasNext() →", it.hasNext())

trace_iterator([7, 3, 15, None, None, 9, 20])

khởi tạo: ngăn xếp = [7, 3]
next() →  3 | pop 3, đẩy nhánh trái của cây con phải → ngăn xếp = [7]
next() →  7 | pop 7, đẩy nhánh trái của cây con phải → ngăn xếp = [15, 9]
next() →  9 | pop 9, đẩy nhánh trái của cây con phải → ngăn xếp = [15]
next() → 15 | pop 15, đẩy nhánh trái của cây con phải → ngăn xếp = [20]
next() → 20 | pop 20, đẩy nhánh trái của cây con phải → ngăn xếp = []
hasNext() → False


# 7. Phiên bản đúng format LeetCode

In [16]:
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        self._push_left(root)

    def _push_left(self, node: Optional[TreeNode]) -> None:
        while node:
            self.stack.append(node)
            node = node.left

    def next(self) -> int:
        node = self.stack.pop()
        self._push_left(node.right)
        return node.val

    def hasNext(self) -> bool:
        return len(self.stack) > 0

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def run_ops(cls, values, ops):
    it, out = None, []
    for op in ops:
        if op == "BSTIterator":
            it = cls(build_tree(values)); out.append(None)
        else:
            out.append(getattr(it, op)())
    return out


ops = ["BSTIterator", "next", "next", "hasNext", "next", "hasNext", "next", "hasNext", "next", "hasNext"]
check("Ví dụ", run_ops(BSTIterator, [7, 3, 15, None, None, 9, 20], ops), [None, 3, 7, True, 9, True, 15, True, 20, False])
check("Một nút", run_ops(BSTIterator, [1], ["BSTIterator", "hasNext", "next", "hasNext"]), [None, True, 1, False])


def bst_from_sorted(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    m = (lo + hi) // 2
    return TreeNode(a[m], bst_from_sorted(a, lo, m - 1), bst_from_sorted(a, m + 1, hi))


def random_bst(n):
    root = None
    def insert(node, x):
        if node is None:
            return TreeNode(x)
        if x < node.val: node.left = insert(node.left, x)
        else: node.right = insert(node.right, x)
        return node
    for x in random.sample(range(10**6), n):
        root = insert(root, x)
    return root


random.seed(173)
for _ in range(1000):
    r = random_bst(random.randint(1, 40))
    it = BSTIterator(r)
    got = []
    while it.hasNext():
        got.append(it.next())
    assert got == inorder_list(r)
print("1000 BST ngẫu nhiên: thứ tự trả về đúng inorder")

big = bst_from_sorted(list(range(10**5)))
it = BSTIterator(big)
max_stack = len(it.stack)
t0 = time.time()
while it.hasNext():
    it.next()
    max_stack = max(max_stack, len(it.stack))
t1 = time.time()
print(f"cây cân bằng 10^5 nút: duyệt hết trong {t1 - t0:.3f}s, ngăn xếp lớn nhất = {max_stack} nút (h ≈ 17)")

Ví dụ            PASS  [None, 3, 7, True, 9, True, 15, True, 20, False]
Một nút          PASS  [None, True, 1, False]


1000 BST ngẫu nhiên: thứ tự trả về đúng inorder

cây cân bằng 10^5 nút: duyệt hết trong 0.126s, ngăn xếp lớn nhất = 16 nút (h ≈ 17)


# 8. Vì sao đúng? Độ phức tạp?

**Đúng:** các lần gọi `next()` thực hiện đúng các bước của inorder lặp, chỉ khác là bị chia ra theo từng lần gọi. Bất biến: ngăn xếp gồm các nút có cây con trái **đã** được duyệt xong và bản thân chúng **chưa** được trả về, xếp theo độ sâu; đỉnh là nút nhỏ nhất chưa trả về.

**Khấu hao `O(1)`:** một lần `next()` có thể đẩy nhiều nút (đi trái cả một nhánh), nhưng **mỗi nút được đẩy đúng một lần và lấy ra đúng một lần** trong toàn bộ quá trình. `n` lần `next()` tổng cộng `O(n)` thao tác → trung bình `O(1)` mỗi lần.

```text
next():    trung bình O(1), xấu nhất O(h)
hasNext(): O(1)
Bộ nhớ:    O(h)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Sau khi pop, quên đẩy nhánh trái của cây con **phải**

In [18]:
class WrongNoRight:
    def __init__(self, root):
        self.st = []
        while root:
            self.st.append(root); root = root.left
    def next(self):
        return self.st.pop().val                      # SAI: bỏ mất cây con phải
    def hasNext(self):
        return bool(self.st)

it_w = WrongNoRight(build_tree([7, 3, 15, None, None, 9, 20]))
got_w = []
while it_w.hasNext():
    got_w.append(it_w.next())
print("Sai :", got_w, "| Đúng:", inorder_list(build_tree([7, 3, 15, None, None, 9, 20])))

Sai : [3, 7] | Đúng: [3, 7, 9, 15, 20]


## Lỗi 2: Chỉ đẩy **con phải** (không đi trái hết mức)

In [19]:
class WrongOnlyRight(WrongNoRight):
    def next(self):
        node = self.st.pop()
        if node.right:
            self.st.append(node.right)                # SAI: phải đẩy cả nhánh trái của node.right
        return node.val

it_w2 = WrongOnlyRight(build_tree([7, 3, 15, None, None, 9, 20]))
got_w2 = []
while it_w2.hasNext():
    got_w2.append(it_w2.next())
print("Sai :", got_w2, "| Đúng:", inorder_list(build_tree([7, 3, 15, None, None, 9, 20])))

Sai : [3, 7, 15, 20] | Đúng: [3, 7, 9, 15, 20]


Sau khi trả `7`, bản sai chỉ đẩy `15` rồi trả luôn `15` — số `9` (con trái của `15`) không bao giờ được đẩy vào ngăn xếp nên bị **mất hẳn**.

## Lỗi 3: `hasNext()` kiểm tra `node.right` thay vì ngăn xếp

`hasNext` phải trả lời "còn phần tử chưa trả về không" — chính là "ngăn xếp có rỗng không". Kiểm tra nút hiện tại có con phải sẽ sai khi phần tử còn lại nằm ở **tổ tiên** (ví dụ sau khi trả `3`, nút `3` không có con phải nhưng vẫn còn `7, 9, 15, 20`).

## Lỗi 4: Đệ quy sâu trên cây lệch

Bản baseline dùng `inorder_list` đệ quy; cây lệch `10^5` nút vượt giới hạn đệ quy mặc định (khoảng 1000).

In [20]:
skewed = None
for v in range(3000, 0, -1):
    skewed = TreeNode(v, None, skewed)               # cây lệch phải 3000 nút
try:
    BSTIteratorList(skewed)
except RecursionError:
    print("RecursionError với cây lệch 3000 nút (bản lưu list dùng đệ quy)")
it_sk = BSTIterator(skewed)
c_sk = 0
while it_sk.hasNext():
    it_sk.next(); c_sk += 1
print("Bản ngăn xếp duyệt được", c_sk, "nút")

RecursionError với cây lệch 3000 nút (bản lưu list dùng đệ quy)
Bản ngăn xếp duyệt được 3000 nút


# 10. Mẹo nhận diện pattern

- **Thiết kế iterator / "trả từng phần tử theo yêu cầu"** → giữ **trạng thái tạm dừng** của thuật toán duyệt (ở đây là ngăn xếp của inorder lặp).
- **Bộ nhớ `O(h)` cho bài cây** → ngăn xếp chứa một đường đi từ gốc.
- **Phân tích khấu hao**: đếm tổng số lần mỗi phần tử được đưa vào/lấy ra.

Bài liên quan: **#94 Binary Tree Inorder Traversal** (Easy), **#284 Peeking Iterator**, **#341 Flatten Nested List Iterator**, **#1586 BST Iterator II**, **#1305 All Elements in Two BSTs**, **#653 Two Sum IV**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với cây `[5, 3, 8, 1, 4, None, 9]`, ghi nội dung ngăn xếp sau khởi tạo và sau mỗi `next()`.

## Bài 2 — Điền chỗ trống

In [21]:
class Exercise2BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        # TODO: đẩy nhánh trái của root

    def next(self) -> int:
        # TODO: pop, đẩy nhánh trái của cây con phải, trả giá trị
        pass

    def hasNext(self) -> bool:
        return len(self.stack) > 0


# assert run_ops(Exercise2BSTIterator, [7, 3, 15, None, None, 9, 20], ["BSTIterator", "next", "next", "hasNext", "next", "hasNext", "next", "hasNext", "next", "hasNext"]) == [None, 3, 7, True, 9, True, 15, True, 20, False]

## Bài 3 — Biến thể: iterator **giảm dần**

Trả các giá trị từ **lớn tới nhỏ** với bộ nhớ `O(h)`. Đối xứng: đẩy nhánh **phải**, sau khi pop thì đẩy nhánh phải của cây con **trái**.

In [22]:
class Exercise3ReverseIterator:
    def __init__(self, root: Optional[TreeNode]):
        # TODO
        pass

    def next(self) -> int:
        pass

    def hasNext(self) -> bool:
        pass


# r3 = Exercise3ReverseIterator(build_tree([7, 3, 15, None, None, 9, 20])); assert [r3.next() for _ in range(5)] == [20, 15, 9, 7, 3] and not r3.hasNext()

## Bài 4 — Biến thể: thêm `peek()` (xem phần tử kế tiếp mà không dịch con trỏ)

In [23]:
class Exercise4PeekIterator(BSTIterator):
    def peek(self) -> int:
        # TODO
        pass


# p4 = Exercise4PeekIterator(build_tree([7, 3, 15, None, None, 9, 20])); assert p4.peek() == 3 and p4.next() == 3 and p4.peek() == 7 and p4.peek() == 7 and p4.next() == 7

## Bài 5 — Mini challenge: All Elements in Two BSTs (LeetCode #1305)

Trả về list tăng dần gồm mọi giá trị của **hai** BST. Dùng **hai iterator** (mỗi cái `O(h)` bộ nhớ) và trộn như merge sort.

In [24]:
def exercise_5_get_all_elements(root1: Optional[TreeNode], root2: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# assert exercise_5_get_all_elements(build_tree([2, 1, 4]), build_tree([1, 0, 3])) == [0, 1, 1, 2, 3, 4]
# assert exercise_5_get_all_elements(None, build_tree([5, 1, 7])) == [1, 5, 7]
# assert exercise_5_get_all_elements(build_tree([1, None, 8]), build_tree([8, 1])) == [1, 1, 8, 8]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_iterator([5, 3, 8, 1, 4, None, 9])` sau khi tự làm.

**Bài 3**
- Mức 1: đổi mọi `left` ↔ `right` trong lời giải chính.

**Bài 4**
- Mức 1: phần tử kế tiếp chính là `self.stack[-1].val` — không cần pop.

**Bài 5**
- Mức 1: hai `BSTIterator` (cây rỗng → `hasNext()` là `False` ngay).
- Mức 2: dùng `peek` (bài 4) để so hai đỉnh, lấy cái nhỏ hơn.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
trace_iterator([5, 3, 8, 1, 4, None, 9])

khởi tạo: ngăn xếp = [5, 3, 1]
next() →  1 | pop 1, đẩy nhánh trái của cây con phải → ngăn xếp = [5, 3]
next() →  3 | pop 3, đẩy nhánh trái của cây con phải → ngăn xếp = [5, 4]
next() →  4 | pop 4, đẩy nhánh trái của cây con phải → ngăn xếp = [5]
next() →  5 | pop 5, đẩy nhánh trái của cây con phải → ngăn xếp = [8]
next() →  8 | pop 8, đẩy nhánh trái của cây con phải → ngăn xếp = [9]
next() →  9 | pop 9, đẩy nhánh trái của cây con phải → ngăn xếp = []
hasNext() → False


## Đáp án Bài 2

In [26]:
class Answer2BSTIterator(BSTIterator):
    pass

assert run_ops(Answer2BSTIterator, [7, 3, 15, None, None, 9, 20], ["BSTIterator", "next", "next", "hasNext", "next", "hasNext", "next", "hasNext", "next", "hasNext"]) == [None, 3, 7, True, 9, True, 15, True, 20, False]
print("Bài 2 PASS (giống BSTIterator ở mục 7)")

Bài 2 PASS (giống BSTIterator ở mục 7)


## Đáp án Bài 3

In [27]:
class Answer3ReverseIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        self._push_right(root)

    def _push_right(self, node):
        while node:
            self.stack.append(node)
            node = node.right

    def next(self) -> int:
        node = self.stack.pop()
        self._push_right(node.left)
        return node.val

    def hasNext(self) -> bool:
        return bool(self.stack)

r3 = Answer3ReverseIterator(build_tree([7, 3, 15, None, None, 9, 20])); assert [r3.next() for _ in range(5)] == [20, 15, 9, 7, 3] and not r3.hasNext()
for _ in range(500):
    r = random_bst(random.randint(1, 30))
    it = Answer3ReverseIterator(r)
    got = []
    while it.hasNext():
        got.append(it.next())
    assert got == inorder_list(r)[::-1]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
class Answer4PeekIterator(BSTIterator):
    def peek(self) -> int:
        return self.stack[-1].val

p4 = Answer4PeekIterator(build_tree([7, 3, 15, None, None, 9, 20])); assert p4.peek() == 3 and p4.next() == 3 and p4.peek() == 7 and p4.peek() == 7 and p4.next() == 7
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [29]:
def answer_5_get_all_elements(root1: Optional[TreeNode], root2: Optional[TreeNode]) -> List[int]:
    a, b = Answer4PeekIterator(root1), Answer4PeekIterator(root2)
    out = []
    while a.hasNext() or b.hasNext():
        if not b.hasNext() or (a.hasNext() and a.peek() <= b.peek()):
            out.append(a.next())
        else:
            out.append(b.next())
    return out

assert answer_5_get_all_elements(build_tree([2, 1, 4]), build_tree([1, 0, 3])) == [0, 1, 1, 2, 3, 4]
assert answer_5_get_all_elements(None, build_tree([5, 1, 7])) == [1, 5, 7]
assert answer_5_get_all_elements(build_tree([1, None, 8]), build_tree([8, 1])) == [1, 1, 8, 8]
for _ in range(500):
    r1, r2 = random_bst(random.randint(1, 20)), random_bst(random.randint(1, 20))
    assert answer_5_get_all_elements(r1, r2) == sorted(inorder_list(r1) + inorder_list(r2))
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Iterator inorder `O(h)` bộ nhớ: ngăn xếp chứa "nhánh trái". Khởi tạo đẩy nhánh trái của gốc; `next()` pop đỉnh, đẩy nhánh trái của cây con phải, trả giá trị; `hasNext()` là "ngăn xếp không rỗng". Mỗi nút vào/ra ngăn xếp đúng một lần → `next()` trung bình `O(1)`.

---

## Checklist trước khi nộp

- [ ] Khởi tạo đẩy **cả** nhánh trái của gốc?
- [ ] `next()` đẩy nhánh trái của `node.right` (không chỉ `node.right`)?
- [ ] `hasNext()` dựa vào ngăn xếp?
- [ ] Không lưu toàn bộ cây vào list?